In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    将文本转为小写，去除非字母/空格字符，分词，构建词汇表（按频率降序分配ID），
    并用滑动窗口生成长度为 n 的特征序列和对应的下一个词标签（忽略无后续词的窗口）。
    
    返回：
        word2idx : dict，词到ID的映射
        features : list of list of int，每个窗口的词ID列表
        labels   : list of int，每个窗口下一个词的ID
    """
    # 1. 小写，保留字母和空格
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # 只保留小写字母和空格
    # 2. 按空格分词（多个空格合并）
    words = text.split()
    if not words:
        return {}, [], []
    
    # 3. 构建词汇表（按频率排序，分配ID从0开始）
    freq = Counter(words)
    # 按频率降序，频率相同按字母序（可选）
    sorted_vocab = sorted(freq.items(), key=lambda x: (-x[1], x[0]))
    word2idx = {word: idx for idx, (word, _) in enumerate(sorted_vocab)}
    
    # 4. 生成特征和标签
    features = []
    labels = []
    for i in range(len(words) - n):  # 窗口起始位置，需保证有后续词作为标签
        window = words[i:i+n]
        next_word = words[i+n]
        features.append([word2idx[w] for w in window])
        labels.append(word2idx[next_word])
    
    return word2idx, features, labels

# 测试
if __name__ == "__main__":
    text = "The time machine"
    n = 2
    vocab, feats, labs = preprocess_text(text, n)
    print("词汇表:", vocab)
    print("特征 (ID序列):", feats)
    print("标签 (ID):", labs)
    # 若想查看原始词，可反向映射
    idx2word = {v: k for k, v in vocab.items()}
    print("特征 (词):", [[idx2word[i] for i in seq] for seq in feats])
    print("标签 (词):", [idx2word[l] for l in labs])

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征 (ID序列): [[1, 2]]
标签 (ID): [0]
特征 (词): [['the', 'time']]
标签 (词): ['machine']


In [2]:
import numpy as np

def rnn_step_forward(x, prev_h, W_hx, W_hh, b_h):
    """
    前向传播：h_t = tanh(W_hx * x + W_hh * prev_h + b_h)
    输入：
        x      : (batch_size, input_size)
        prev_h : (batch_size, hidden_size)
        W_hx   : (input_size, hidden_size)
        W_hh   : (hidden_size, hidden_size)
        b_h    : (hidden_size,)
    返回：
        h_next : (batch_size, hidden_size)
        缓存    : (x, prev_h, W_hx, W_hh, b_h, h_next) 用于反向
    """
    h_next = np.tanh(np.dot(x, W_hx) + np.dot(prev_h, W_hh) + b_h)
    cache = (x, prev_h, W_hx, W_hh, b_h, h_next)
    return h_next, cache

def rnn_step_backward(dh_next, cache):
    """
    单步反向传播，已知损失对 h_next 的梯度 dh_next（形状同 h_next）
    计算 dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    x, prev_h, W_hx, W_hh, b_h, h_next = cache
    # tanh 的导数：1 - tanh^2
    dtanh = dh_next * (1 - h_next ** 2)
    
    # 分别对各项求导
    dx = np.dot(dtanh, W_hx.T)
    dh_prev = np.dot(dtanh, W_hh.T)
    dW_hx = np.dot(x.T, dtanh)
    dW_hh = np.dot(prev_h.T, dtanh)
    db_h = np.sum(dtanh, axis=0)  # 对 batch 求和
    
    return dx, dh_prev, dW_hx, dW_hh, db_h

# 测试：随机数据验证梯度形状
if __name__ == "__main__":
    batch_size, input_size, hidden_size = 3, 4, 5
    x = np.random.randn(batch_size, input_size)
    prev_h = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    h_next, cache = rnn_step_forward(x, prev_h, W_hx, W_hh, b_h)
    dh_next = np.random.randn(batch_size, hidden_size)
    dx, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)
    
    print("前向输出形状:", h_next.shape)
    print("dx形状:", dx.shape)
    print("dh_prev形状:", dh_prev.shape)
    print("dW_hx形状:", dW_hx.shape)
    print("dW_hh形状:", dW_hh.shape)
    print("db_h形状:", db_h.shape)

前向输出形状: (3, 5)
dx形状: (3, 4)
dh_prev形状: (3, 5)
dW_hx形状: (4, 5)
dW_hh形状: (5, 5)
db_h形状: (5,)


In [3]:
import torch
import torch.nn as nn

def bidirectional_rnn_encoder(X, hidden_dim, num_layers=1, use_torch=True):
    """
    接收序列 X (seq_len, batch, input_dim)，返回每个时间步拼接后的隐藏状态
    和最终时间步的拼接状态（作为序列表示）。
    若 use_torch=True 使用 torch.nn.RNN，否则手动实现（这里只提供 torch 版本）。
    """
    seq_len, batch, input_dim = X.shape
    
    if use_torch:
        # 使用 PyTorch 的 RNN，bidirectional=True
        rnn = nn.RNN(input_size=input_dim, hidden_size=hidden_dim, 
                     num_layers=num_layers, batch_first=False, bidirectional=True)
        # 初始化隐藏状态 (num_layers * 2, batch, hidden_dim)
        h0 = torch.zeros(num_layers * 2, batch, hidden_dim)
        output, h_n = rnn(X, h0)  # output: (seq_len, batch, hidden_dim*2)
        # output 已经是每个时间步前向+后向拼接后的状态
        # 最终时间步的拼接状态：取最后一个时间步，即 output[-1, :, :] 形状 (batch, 2*hidden_dim)
        final_state = output[-1, :, :]
        return output, final_state
    else:
        # 手动实现双向 RNN（略，但为完整性可提供简单实现）
        raise NotImplementedError("手动实现请参考 torch 版本")

# 测试
if __name__ == "__main__":
    seq_len, batch, input_dim = 5, 2, 3
    hidden_dim = 4
    X = torch.randn(seq_len, batch, input_dim)
    output, final = bidirectional_rnn_encoder(X, hidden_dim, num_layers=1, use_torch=True)
    print("output 形状 (seq_len, batch, 2*hidden):", output.shape)  # (5,2,8)
    print("final 形状 (batch, 2*hidden):", final.shape)            # (2,8)

output 形状 (seq_len, batch, 2*hidden): torch.Size([5, 2, 8])
final 形状 (batch, 2*hidden): torch.Size([2, 8])


In [4]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, W, W_out):
    """
    输入：
        context_indices : list of list of int，每个样本有 context_size 个上下文词索引
                         shape: (batch_size, context_size)
        W               : 输入权重矩阵 (V, d)
        W_out           : 输出权重矩阵 (d, V)
    返回：
        loss : 标量张量，平均交叉熵损失
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    # 1. 获取每个上下文词的嵌入向量 (batch, context_size, d)
    emb = W[context_indices]  # (batch, context_size, d)
    # 2. 平均得到隐藏层 (batch, d)
    h = emb.mean(dim=1)  # (batch, d)
    # 3. 计算输出 logits (batch, V)
    logits = torch.matmul(h, W_out)  # (batch, V)
    # 4. 目标中心词索引 (假设是每个样本的第一个词？题目未明确，但通常CBOW预测中心词)
    #    这里假设 context_indices 的最后一列是中心词？但实际上CBOW是给定上下文预测中心词。
    #    题目说"给定一批上下文词的索引列表（每个样本有 context_size 个上下文词）"，
    #    目标为中心词索引，但未提供目标输入。因此我假设目标中心词就是每个样本的最后一个词（作为预测目标），
    #    或者额外传入 target_indices。为了完整性，我额外添加一个参数 target_indices，
    #    但题目要求只传入 context_indices，所以我们只能假设 context_indices 中包含了中心词？
    #    典型CBOW训练：输入上下文词，输出中心词，因此需要单独的目标。
    #    这里为了符合题目描述，我修改函数签名，增加 target_indices。
    #    但题目说"给定一批上下文词的索引列表...目标为中心词索引"，意味着函数应接收目标和上下文，
    #    所以重新设计如下。
    #    为了不混淆，我重新实现一个函数，接收 context_indices 和 target_indices。
    pass

# 根据题目要求，我们改为以下实现（明确接收目标）
def cbow_loss(context_indices, target_indices, W, W_out):
    """
    context_indices : (batch_size, context_size) 上下文词索引
    target_indices  : (batch_size,) 中心词索引
    W, W_out        : 输入和输出权重
    返回损失值（标量）
    """
    batch_size, context_size = context_indices.shape
    # 嵌入平均
    emb = W[context_indices]  # (batch, context_size, d)
    h = emb.mean(dim=1)       # (batch, d)
    logits = torch.matmul(h, W_out)  # (batch, V)
    # 交叉熵损失
    loss = F.cross_entropy(logits, target_indices)
    return loss

# 测试
if __name__ == "__main__":
    V, d = 10, 3
    batch_size, context_size = 4, 2
    W = torch.randn(V, d, requires_grad=True)
    W_out = torch.randn(d, V, requires_grad=True)
    context = torch.randint(0, V, (batch_size, context_size))
    targets = torch.randint(0, V, (batch_size,))
    loss = cbow_loss(context, targets, W, W_out)
    print("CBOW 损失值:", loss.item())

CBOW 损失值: 1.676938533782959


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def multi_head_attention_forward(X, num_heads, d_model, W_q, W_k, W_v, W_o):
    """
    手动实现多头注意力的前向传播（无batch维度? 但 X 形状为 (seq_len, batch, d_model)）
    为了简单，我们按 batch 独立处理，使用 PyTorch 的矩阵乘法。
    参数：
        X : (seq_len, batch, d_model)
        num_heads : 头数
        d_model : 模型维度，需能被 num_heads 整除
        W_q, W_k, W_v, W_o : 线性投影权重，形状分别为 (d_model, d_model) 等（无偏置）
    返回：
        output : (seq_len, batch, d_model)
    """
    seq_len, batch, d_model = X.shape
    assert d_model % num_heads == 0
    d_k = d_model // num_heads
    d_v = d_k  # 通常相等
    
    # 线性投影，得到 Q, K, V (形状均为 seq_len, batch, d_model)
    Q = torch.matmul(X, W_q)  # (seq_len, batch, d_model)
    K = torch.matmul(X, W_k)
    V = torch.matmul(X, W_v)
    
    # 变形为 (seq_len, batch, num_heads, d_k) 并交换维度为 (batch, num_heads, seq_len, d_k)
    Q = Q.view(seq_len, batch, num_heads, d_k).transpose(0, 1).transpose(1, 2)  # (batch, num_heads, seq_len, d_k)
    K = K.view(seq_len, batch, num_heads, d_k).transpose(0, 1).transpose(1, 2)
    V = V.view(seq_len, batch, num_heads, d_v).transpose(0, 1).transpose(1, 2)
    
    # 缩放点积注意力
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)  # (batch, num_heads, seq_len, seq_len)
    attn_weights = F.softmax(scores, dim=-1)
    attn_output = torch.matmul(attn_weights, V)  # (batch, num_heads, seq_len, d_v)
    
    # 合并头：转置回 (seq_len, batch, num_heads, d_v) 再 reshape
    attn_output = attn_output.transpose(1, 2).transpose(0, 1)  # (seq_len, batch, num_heads, d_v)
    attn_output = attn_output.reshape(seq_len, batch, d_model)  # (seq_len, batch, d_model)
    
    # 最终线性层
    output = torch.matmul(attn_output, W_o)  # (seq_len, batch, d_model)
    return output

# 测试
if __name__ == "__main__":
    seq_len, batch, d_model = 5, 2, 4
    num_heads = 2
    # 初始化权重（无偏置）
    W_q = torch.randn(d_model, d_model)
    W_k = torch.randn(d_model, d_model)
    W_v = torch.randn(d_model, d_model)
    W_o = torch.randn(d_model, d_model)
    
    X = torch.randn(seq_len, batch, d_model)
    output = multi_head_attention_forward(X, num_heads, d_model, W_q, W_k, W_v, W_o)
    print("多头注意力输出形状:", output.shape)  # (5,2,4)

多头注意力输出形状: torch.Size([5, 2, 4])
